# 03. Implementación de un Transformer desde cero en PyTorch

En este notebook escalamos en complejidad técnica construyendo un **Mini-Transformer**. A diferencia del modelo anterior (GRU), aquí no dependemos de la recurrencia, sino del mecanismo de **Self-Attention** para capturar relaciones entre palabras sin importar su distancia en el texto.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score
from datasets import load_dataset
from transformers import AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)
print(f'Dispositivo de ejecución: {DEVICE}')

### 1. Definición de la Arquitectura del Transformer

A continuación, definimos la clase `SimpleTransformer`. A diferencia de una capa densa estándar, utilizamos:
* **MultiheadAttention**: Permite que el modelo se enfoque en diferentes partes de la frase simultáneamente.
* **LayerNorm**: Normaliza las activaciones para estabilizar el entrenamiento.
* **Masked Mean Pooling**: Una técnica crítica para ignorar los tokens de relleno (padding) al calcular el vector resumen de la frase.

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_classes, pad_id=0, dropout=0.3):
        """
        Constructor del Transformer.
        - vocab_size: Tamaño del vocabulario del tokenizador.
        - embed_dim: Dimensión de los vectores de embedding.
        - num_heads: Número de cabezas de atención paralelas.
        - pad_id: Índice del token de padding para ser ignorado.
        """
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=dropout)
        self.ln = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    @staticmethod
    def masked_mean(x, attention_mask, eps=1e-9):
        """
        Calcula el promedio de los embeddings ignorando los tokens de padding.
        - x: Tensores de salida de la atención [Batch, Seq_Len, Embed_Dim]
        - attention_mask: Máscara binaria (1 para tokens reales, 0 para padding).
        """
        mask = attention_mask.unsqueeze(-1).expand(x.size()).float()
        return (x * mask).sum(dim=1) / (mask.sum(dim=1) + eps)

    def forward(self, input_ids, attention_mask):
        """
        Flujo de datos del modelo.
        1. Generación de embeddings.
        2. Aplicación de Self-Attention con máscara de padding.
        3. Conexión residual y normalización (Add & Norm).
        4. Pooling y clasificación final.
        """
        x = self.embedding(input_ids)
        
        # MultiheadAttention requiere una máscara booleana invertida (True donde hay padding)
        key_padding_mask = (attention_mask == 0)
        attn_output, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        
        # Conexión residual y normalización
        x = self.ln(x + attn_output)
        
        # Resumen de la secuencia e inferencia
        x = self.masked_mean(x, attention_mask)
        return self.fc(self.dropout(x))

**Explicación técnica:**
- **`__init__`**: Inicializa los pesos aprendibles. La capa de `embedding` se entrena desde cero en este ejercicio.
- **`masked_mean`**: Evita que el "ruido" de los ceros del padding altere el promedio semántico de la frase. Es fundamental en modelos que procesan batches de longitud variable.
- **`forward`**: Implementa la lógica de "atención a uno mismo". Cada palabra se redefine en función de su contexto en la frase.

### 2. Procesamiento con Tokenizador de Hugging Face

Utilizamos el tokenizador de `bert-base-uncased` por su robustez, aunque el modelo de clasificación sea nuestro propio diseño.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def make_collate_fn_transformer(tokenizer_fn, max_len=128):
    """
    Crea una función collate que tokeniza en tiempo real durante la carga de batches.
    """
    def collate(batch):
        texts, labels = zip(*[(d["text"], d["labels"]) for d in batch])
        enc = tokenizer_fn(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"], torch.tensor(labels)
    return collate

# Preparamos DataLoaders
from datasets import load_dataset
# (Se asume que el dataset ya está filtrado a Ekman como en notebooks anteriores)
collate_fn = make_collate_fn_transformer(tokenizer)
# train_loader = DataLoader(ekman_dataset['train'], batch_size=64, shuffle=True, collate_fn=collate_fn)

**Explicación técnica:**
- **`make_collate_fn_transformer`**: Actúa como un puente entre los datos crudos y el modelo. Convierte texto en `input_ids` (números) y crea la `attention_mask` que el Transformer necesita para saber qué tokens ignorar.

### 3. Entrenamiento y Evaluación

Implementamos el bucle de entrenamiento estándar de PyTorch con soporte para validación.

In [ ]:
def train_transformer(model, train_loader, val_loader, epochs=10):
    """
    Ejecuta el ciclo de entrenamiento y validación del Transformer.
    """
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for ids, mask, labels in train_loader:
            ids, mask, labels = ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(ids, mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} completada. Loss: {total_loss/len(train_loader):.4f}")

## Reflexión Técnica: ¿Por qué Atención?

A diferencia del modelo GRU, el Transformer logra un **Accuracy de ~59.4%** y un **F1-Score superior**. Esto se debe a que el mecanismo de atención no procesa la información de forma secuencial, lo que evita la pérdida de información en frases largas y permite que el modelo entienda mejor el contexto global de los comentarios de Reddit.